# 🧑‍🎨 Person Outfit & Location Generator

Upload **one clear photo of a face** and get several images of that **same person**
in different **clothing** and **locations** — completely free, running on Google
Colab's free cloud GPU. Nothing to install on your own computer.

**How to run it (takes ~5–8 minutes the first time):**
1. Click **Runtime ▸ Change runtime type** and pick a **GPU** (the free **T4** is fine). Save.
2. Run every cell top to bottom (**Runtime ▸ Run all**, or press ▶ on each cell in order).
3. When the last cell finishes it prints a public **`https://…gradio.live`** link — open it,
   upload a face photo, and click **Generate**.

Under the hood this uses [InstantID](https://github.com/InstantID/InstantID) (open source)
to keep the face consistent while changing the outfit and background.

> The free Colab GPU has usage limits and sessions time out after a while — just re-run
> the cells to start a fresh session. It's free every time.


### 1. Check you have a GPU

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU found! Go to Runtime > Change runtime type > select 'T4 GPU', "
    "click Save, then run this cell again."
)
print("GPU ready:", torch.cuda.get_device_name(0))

### 2. Install the software (one-time, ~2 min)

In [ ]:
# Pinned to a known-good combination for InstantID.
%pip install -q "huggingface_hub==0.20.2" "diffusers==0.25.1" "transformers==4.37.2" \
    "accelerate==0.27.2" "peft==0.8.2" safetensors einops omegaconf
%pip install -q "insightface==0.7.3" onnxruntime-gpu opencv-python-headless
%pip install -q "gradio==4.44.1"
print("Done installing. Dependency warnings above are normal and safe to ignore.")

### 3. Download the open-source models (~3 min, first time only)

In [ ]:
import os
# InstantID pipeline code
if not os.path.exists("InstantID"):
    !git clone -q https://github.com/InstantID/InstantID.git

# Model weights from Hugging Face (public, free)
from huggingface_hub import hf_hub_download
os.makedirs("checkpoints", exist_ok=True)
hf_hub_download(repo_id="InstantX/InstantID", filename="ControlNetModel/config.json", local_dir="./checkpoints")
hf_hub_download(repo_id="InstantX/InstantID", filename="ControlNetModel/diffusion_pytorch_model.safetensors", local_dir="./checkpoints")
hf_hub_download(repo_id="InstantX/InstantID", filename="ip-adapter.bin", local_dir="./checkpoints")
print("Models downloaded.")

### 4. Load the model into memory (~1–2 min)

In [ ]:
import sys, cv2, torch, numpy as np
from PIL import Image
sys.path.append("./InstantID")

from insightface.app import FaceAnalysis
from diffusers.models import ControlNetModel
from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline, draw_kps

# Face encoder — the antelopev2 model auto-downloads the first time this runs.
face_app = FaceAnalysis(name="antelopev2", root="./",
                        providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
face_app.prepare(ctx_id=0, det_size=(640, 640))

controlnet = ControlNetModel.from_pretrained("./checkpoints/ControlNetModel", torch_dtype=torch.float16)
pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
    "wangqixun/YamerMIX_v8", controlnet=controlnet, torch_dtype=torch.float16)
pipe.cuda()
pipe.load_ip_adapter_instantid("./checkpoints/ip-adapter.bin")
pipe.enable_vae_tiling()
print("Model loaded — ready to generate!")

### 5. The outfit & location ideas + the generator

This is the "surprise me" logic: it randomly pairs an outfit with a location for each
image, so every run gives a fresh mix. Want your own styles? Edit the `OUTFITS` and
`LOCATIONS` lists below and re-run this cell.

In [ ]:
import random

OUTFITS = [
    "a sharp tailored navy business suit",
    "a cozy cream chunky-knit sweater and blue jeans",
    "elegant formal evening wear",
    "casual streetwear with a leather jacket and sneakers",
    "athletic sportswear",
    "a warm winter coat with a wool scarf",
    "a light linen summer outfit",
    "smart-casual clothes with a wool blazer",
    "a stylish beige trench coat",
    "vintage 1970s fashion",
    "bohemian festival clothing",
    "a chic all-white outfit",
]

LOCATIONS = [
    "on a busy neon-lit Tokyo street at night",
    "on a sunny tropical beach with palm trees",
    "at a cozy Parisian sidewalk cafe",
    "in a sleek modern office with glass walls",
    "in a lush green forest with soft sunlight",
    "on a snowy mountain overlook",
    "in a bright contemporary art gallery",
    "at a rooftop bar during golden-hour sunset",
    "in a grand old library full of books",
    "walking through a colorful flower field",
    "on a rainy city street with glowing reflections",
    "in a warm desert landscape at dusk",
]

STYLE = ("photorealistic portrait, natural lighting, shot on a 50mm lens, "
         "sharp focus, highly detailed, professional photography")
NEG = ("(lowres, low quality, worst quality:1.2), cartoon, anime, illustration, painting, "
       "drawing, 3d render, deformed, disfigured, extra fingers, bad hands, bad anatomy, "
       "watermark, text, jpeg artifacts")


def make_prompts(n, subject):
    combos = [(o, l) for o in OUTFITS for l in LOCATIONS]
    random.shuffle(combos)
    out = []
    for outfit, loc in combos[:n]:
        prompt = f"{STYLE} of a {subject} wearing {outfit}, {loc}"
        out.append((prompt, f"{outfit} — {loc}"))
    return out


def _resize(img, max_side=1024):
    w, h = img.size
    scale = min(max_side / w, max_side / h, 1.0)
    nw, nh = int(w * scale), int(h * scale)
    nw -= nw % 8
    nh -= nh % 8
    return img.resize((max(nw, 64), max(nh, 64)))


def _get_face(pil_img):
    bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    faces = face_app.get(bgr)
    if not faces:
        return None, None
    faces = sorted(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    face = faces[-1]  # largest face
    return face.embedding, face.kps


def generate(face_pil, num_images, subject, id_strength, steps, seed):
    import gradio as gr
    if face_pil is None:
        raise gr.Error("Please upload a photo first.")
    face_pil = _resize(face_pil.convert("RGB"))
    emb, kps = _get_face(face_pil)
    if emb is None:
        raise gr.Error("No face detected — try a clearer, front-facing photo.")
    kps_img = draw_kps(face_pil, kps)
    pipe.set_ip_adapter_scale(float(id_strength))

    base_seed = int(seed) if seed is not None and int(seed) >= 0 else random.randint(0, 2**31 - 1)
    results = []
    for i, (prompt, caption) in enumerate(make_prompts(int(num_images), subject)):
        gen = torch.Generator(device="cuda").manual_seed(base_seed + i)
        image = pipe(
            prompt=prompt,
            negative_prompt=NEG,
            image_embeds=emb,
            image=kps_img,
            controlnet_conditioning_scale=0.8,
            num_inference_steps=int(steps),
            guidance_scale=5.0,
            generator=gen,
        ).images[0]
        results.append((image, caption))
    return results

print("Generator ready.")

### 6. Launch the app — this prints your shareable link

In [ ]:
import gradio as gr

with gr.Blocks(title="Person Outfit & Location Generator") as demo:
    gr.Markdown(
        "# 🧑‍🎨 Same person, new outfits & places\n"
        "Upload one clear, front-facing photo and get several images of that same "
        "person in different clothing and locations."
    )
    with gr.Row():
        with gr.Column(scale=1):
            inp = gr.Image(type="pil", label="Your photo")
            subject = gr.Radio(["person", "woman", "man"], value="person", label="Subject")
            num = gr.Slider(1, 8, value=4, step=1, label="Number of images")
            with gr.Accordion("Advanced settings", open=False):
                id_strength = gr.Slider(0.3, 1.0, value=0.8, step=0.05,
                                        label="Identity strength (higher = more like your photo)")
                steps = gr.Slider(20, 50, value=30, step=1, label="Quality steps (higher = slower)")
                seed = gr.Number(value=-1, label="Seed (-1 = random each time)")
            btn = gr.Button("✨ Generate", variant="primary")
        with gr.Column(scale=2):
            gallery = gr.Gallery(label="Results", columns=2, height=760, object_fit="contain")

    btn.click(generate, [inp, num, subject, id_strength, steps, seed], gallery)

# share=True gives you a public https://...gradio.live link you can open in any browser.
demo.launch(share=True, debug=False)

---
### Troubleshooting

- **"No GPU found"** — Runtime ▸ Change runtime type ▸ **T4 GPU** ▸ Save, then run all cells again.
- **Out of memory (CUDA OOM)** — lower **Number of images**, or in cell 4 replace `pipe.cuda()`
  with `pipe.enable_model_cpu_offload()` (slower but uses less VRAM), then re-run cells 4–6.
- **"No face detected"** — use a clear, well-lit, front-facing photo where the face is not too small.
- **Install/version errors** — Runtime ▸ **Disconnect and delete runtime**, then start fresh and run all cells.
- **The gradio.live link stopped working** — it expires when the Colab session ends. Just re-run
  the last cell (or all cells) to get a new link. Always free.

### Want to change the styles?
Edit the `OUTFITS` and `LOCATIONS` lists in cell 5 and re-run that cell — then generate again.
